In [15]:
import os
import re

import pdfplumber

print("Biblioteki załadowane")


Biblioteki załadowane


In [16]:
PDF_FILENAME = "OC_os_fiz_przy_EDU_Plus_2b489658.pdf"
PDF_PATH = os.path.join("data", "zbior_danych", "samorzad.pk.edu.pl", "2026-01", PDF_FILENAME)

if not os.path.exists(PDF_PATH):
    raise FileNotFoundError(f"Nie znaleziono pliku PDF: {PDF_PATH}")

print(f"Znaleziono plik: {PDF_PATH}")


Znaleziono plik: data\zbior_danych\samorzad.pk.edu.pl\2026-01\OC_os_fiz_przy_EDU_Plus_2b489658.pdf


In [17]:
Full_pdf_text = ""
page_texts = []

print(f"Ekstrakcja tekstu z: {PDF_FILENAME}\n")

with pdfplumber.open(PDF_PATH) as pdf:
    for page_number, page in enumerate(pdf.pages, start=1):
        text = page.extract_text(layout=True)

        if text:
            page_texts.append(text)
            print(f"Strona {page_number}: wyciągnięto {len(text.split())} słów.")
        else:
            print(f"Strona {page_number}: [UWAGA] brak tekstu. Możliwy skan (wymagany OCR).")

Full_pdf_text = "\n".join(page_texts).strip()

print("\nPodgląd pierwszych 300 znaków wyciągniętego tekstu:")
print("-" * 50)
preview = Full_pdf_text[:300]
print(preview + ("\n[...]" if len(Full_pdf_text) > 300 else ""))
print("-" * 50)


Ekstrakcja tekstu z: OC_os_fiz_przy_EDU_Plus_2b489658.pdf

Strona 1: wyciągnięto 618 słów.
Strona 2: wyciągnięto 306 słów.
Strona 3: wyciągnięto 834 słów.
Strona 4: wyciągnięto 128 słów.
Strona 5: wyciągnięto 1149 słów.
Strona 6: wyciągnięto 1207 słów.
Strona 7: wyciągnięto 1114 słów.
Strona 8: wyciągnięto 1182 słów.
Strona 9: wyciągnięto 800 słów.
Strona 10: wyciągnięto 1138 słów.
Strona 11: wyciągnięto 930 słów.
Strona 12: wyciągnięto 1209 słów.
Strona 13: wyciągnięto 1072 słów.
Strona 14: wyciągnięto 492 słów.

Podgląd pierwszych 300 znaków wyciągniętego tekstu:
--------------------------------------------------
Ubezpieczenie Odpowiedzialności cywilnej osób fizycznych                      
                                                                                       
         w życiu prywatnym oraz nauczycieli i dyrektorów placówek                      
         oświatowych w ramach oferty EDU Plus
[...]
--------------------------------------------------


In [18]:
from transformers import AutoTokenizer

model_name = "allegro/herbert-base-cased"

print(f"Pobieranie i ładowanie tokenizera: {model_name}...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

model_max_length = tokenizer.model_max_length
if model_max_length is None or model_max_length > 100_000:
    model_max_length = 512

special_tokens_count = tokenizer.num_special_tokens_to_add(pair=False)
TOKENIZER_CHUNK_SIZE = max(1, model_max_length - special_tokens_count)

print(f"Tokenizer gotowy. Limit treści chunku: {TOKENIZER_CHUNK_SIZE} tokenów.")


Pobieranie i ładowanie tokenizera: allegro/herbert-base-cased...
Tokenizer gotowy. Limit treści chunku: 510 tokenów.


In [19]:
def split_text_into_chunks(text, tokenizer, max_tokens):
    parts = re.findall(r"\s*\S+", text)
    chunks = []
    current_parts = []
    current_token_count = 0

    for part in parts:
        part_token_count = len(tokenizer.tokenize(part))

        if current_parts and current_token_count + part_token_count > max_tokens:
            chunks.append("".join(current_parts))
            current_parts = [part]
            current_token_count = part_token_count
        else:
            current_parts.append(part)
            current_token_count += part_token_count

    if current_parts:
        chunks.append("".join(current_parts))

    return chunks

chunks = split_text_into_chunks(Full_pdf_text, tokenizer, TOKENIZER_CHUNK_SIZE)
chunk_token_counts = [len(tokenizer.tokenize(chunk)) for chunk in chunks]

print(f"Utworzono {len(chunks)} chunków wyłącznie na potrzeby limitu tokenizera.")
print(f"Najdłuższy chunk: {max(chunk_token_counts, default=0)} / {TOKENIZER_CHUNK_SIZE} tokenów.")


Utworzono 53 chunków wyłącznie na potrzeby limitu tokenizera.
Najdłuższy chunk: 510 / 510 tokenów.


In [20]:
tokens = []

for chunk in chunks:
    tokens.extend(tokenizer.tokenize(chunk))

print(f"Tokeny całego PDF-a: {len(tokens)}")
print(tokens[:50])


Tokeny całego PDF-a: 23782
['Ubezpieczenie</w>', 'Odpowiedzi', 'alności</w>', 'cywilnej</w>', 'osób</w>', 'fizycznych</w>', 'w</w>', 'życiu</w>', 'prywatnym</w>', 'oraz</w>', 'nauczycieli</w>', 'i</w>', 'dyrektorów</w>', 'placówek</w>', 'oświatowych</w>', 'w</w>', 'ramach</w>', 'oferty</w>', 'E', 'D', 'U</w>', 'Plus</w>', 'Dokument</w>', 'zawierający</w>', 'informacje</w>', 'o</w>', 'produ', 'kcie</w>', 'ubezpieczeniowym</w>', 'Przedsiębiorstwo</w>', ':</w>', 'Inter', 'Ri', 'sk</w>', 'Towarzystwo</w>', 'Ubezpieczeń</w>', 'Spółka</w>', 'Ak', 'cyjna</w>', 'V', 'ien', 'na</w>', 'In', 'surance</w>', 'Group</w>', 'z</w>', 'siedzibą</w>', 'w</w>', 'Polsce</w>', ',</w>']


In [21]:
tokenization_flow = {
    "Full_pdf_text": Full_pdf_text,
    "chunks": chunks,
    "tokens": tokens,
}

print("Gotowy przepływ:")
print(f"Full_pdf_text: {len(tokenization_flow['Full_pdf_text'])} znaków")
print(f"chunks: {len(tokenization_flow['chunks'])} elementów")
print(f"tokens: {len(tokenization_flow['tokens'])} elementów")


Gotowy przepływ:
Full_pdf_text: 155297 znaków
chunks: 53 elementów
tokens: 23782 elementów


In [22]:
assert isinstance(Full_pdf_text, str)
assert isinstance(chunks, list)
assert isinstance(tokens, list)
assert all(isinstance(chunk, str) for chunk in chunks)
assert all(isinstance(token, str) for token in tokens)
assert sum(chunk_token_counts) == len(tokens)

print("Walidacja struktury OK: Full_pdf_text -> chunks -> tokens")


Walidacja struktury OK: Full_pdf_text -> chunks -> tokens
